In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

print("=========================================================")
print("🚀 DÉFI : GESTION & ANALYSE DES SALAIRES EN SCIENCE DES DONNÉES")
print("=========================================================\n")

# -------------------------------------------------------------------------
# ÉTAPE 1 : Téléchargement et importation du jeu de données
# -------------------------------------------------------------------------
# URL du fichier ds_salaries.csv (Dataset officiel standard)
url_salaries = "https://raw.githubusercontent.com/YuluDuan/Hypothesis-Testing-Data-Science-salary-comparison-in-different-location/main/ds_salaries.csv"
df_salaries = pd.read_csv(url_salaries)

# Nettoyage cosmétique : suppression d'une éventuelle colonne d'index inutile
if 'Unnamed: 0' in df_salaries.columns:
    df_salaries.drop(columns=['Unnamed: 0'], inplace=True)

print("✅ Étape 1 : Données importées avec succès !")
print(f"Structure du dataset : {df_salaries.shape[0]} lignes et {df_salaries.shape[1]} colonnes.\n")
print("Aperçu des 3 premières lignes :")
print(df_salaries[['experience_level', 'job_title', 'salary_in_usd']].head(3))
print("-" * 60)


# -------------------------------------------------------------------------
# ÉTAPE 2 : Normalisation Min-Max du Salaire (Mise à l'échelle [0, 1])
# -------------------------------------------------------------------------
# Nous utilisons 'salary_in_usd' pour avoir une base de comparaison cohérente (Devise unique)
scaler = MinMaxScaler()
df_salaries['salary_normalised'] = scaler.fit_transform(df_salaries[['salary_in_usd']])

print("\n✅ Étape 2 : Normalisation Min-Max appliquée.")
print(df_salaries[['salary_in_usd', 'salary_normalised']].describe().loc[['min', 'max', 'mean']])
print("-" * 60)


# -------------------------------------------------------------------------
# ÉTAPE 3 : Réduction de la dimensionnalité (Analyse en Composantes Principales)
# -------------------------------------------------------------------------
# L'ACP (PCA) ne traite que les données numériques.
# Préparons un sous-ensemble numérique en encodant d'abord les variables catégorielles.
df_numeric = pd.get_dummies(df_salaries.drop(columns=['salary', 'salary_currency']), drop_first=True)

# L'ACP est très sensible aux échelles, on applique le MinMaxScaler sur l'ensemble de notre matrice numérique
df_numeric_scaled = MinMaxScaler().fit_transform(df_numeric)

# Application de l'ACP pour réduire à 2 dimensions (2 composantes principales)
pca = PCA(n_components=2)
composantes_principales = pca.fit_transform(df_numeric_scaled)

# Intégration des composantes dans un nouveau DataFrame
df_pca = pd.DataFrame(data=composantes_principales, columns=['Composante_1', 'Composante_2'])

print("\n✅ Étape 3 : Réduction de dimensionnalité (ACP) terminée.")
print(f"Ancienne dimension (après Encodage One-Hot) : {df_numeric.shape[1]} caractéristiques.")
print(f"Nouvelle dimension (après ACP) : {df_pca.shape[1]} caractéristiques.")
print(f"Variance totale conservée par les 2 composantes : {np.sum(pca.explained_variance_ratio_)*100:.2f}%")
print(df_pca.head(3))
print("-" * 60)


# -------------------------------------------------------------------------
# ÉTAPE 4 : Agrégation par "niveau d'expérience"
# -------------------------------------------------------------------------
# Groupement et calcul simultané de la moyenne et de la médiane
analyse_experience = df_salaries.groupby('experience_level')['salary_in_usd'].agg(['mean', 'median'])

# Optionnel : Renommer les index pour une meilleure lisibilité métier
mapping_experience = {
    'EN': 'Junior / Entry-level',
    'MI': 'Intermédiaire / Mid-level',
    'SE': 'Senior / Expert',
    'EX': 'Directeur / Executive'
}
analyse_experience.index = analyse_experience.index.map(mapping_experience)

# Tri des résultats par salaire moyen croissant
analyse_experience = analyse_experience.sort_values(by='mean')

print("\n✅ Étape 4 : Agrégation des salaires (en USD) par niveau d'expérience :")
print("=" * 65)
print(analyse_experience.round(2))
print("=" * 65)
print("\n🎉 Défi quotidien terminé avec succès !")

🚀 DÉFI : GESTION & ANALYSE DES SALAIRES EN SCIENCE DES DONNÉES

✅ Étape 1 : Données importées avec succès !
Structure du dataset : 607 lignes et 11 colonnes.

Aperçu des 3 premières lignes :
  experience_level                   job_title  salary_in_usd
0               MI              Data Scientist          79833
1               SE  Machine Learning Scientist         260000
2               SE           Big Data Engineer         109024
------------------------------------------------------------

✅ Étape 2 : Normalisation Min-Max appliquée.
      salary_in_usd  salary_normalised
min     2859.000000           0.000000
max   600000.000000           1.000000
mean  112297.869852           0.183271
------------------------------------------------------------

✅ Étape 3 : Réduction de dimensionnalité (ACP) terminée.
Ancienne dimension (après Encodage One-Hot) : 166 caractéristiques.
Nouvelle dimension (après ACP) : 2 caractéristiques.
Variance totale conservée par les 2 composantes : 29.66%
 